<a href="https://colab.research.google.com/github/jeremydouti2-hub/Data-Science-Projects/blob/main/E_Commerce_Product_Intelligence_using_Image_and_Text_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
uploded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv(
    "modular 6_1.csv",
    encoding='latin1',
    sep=',',
    engine='python',
    on_bad_lines='skip'   #  remplace error_bad_lines
)

print(df.head())

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Charger le dataset (train)
data = fetch_20newsgroups(subset='train')

# Afficher un exemple
print(data.data[0])

# Afficher les catégories
print(data.target_names)

# Nombre de documents
print(len(data.data))

In [ ]:
'alt.atheism', 'comp.graphics', ..., 'talk.religion.misc'

In [ ]:
from sklearn.datasets import fetch_20newsgroups

train_data = fetch_20newsgroups(subset='train')
test_data = fetch_20newsgroups(subset='test')

print(len(train_data.data), len(test_data.data))

In [ ]:
# Télécharger le dump Wikipedia (version simple, plus légère)
!wget https://dumps.wikimedia.org/simplewiki/latest/simplewiki-latest-pages-articles.xml.bz2

# Lire une partie du fichier
import bz2

with bz2.open("simplewiki-latest-pages-articles.xml.bz2", "rb") as f:
    data = f.read(1000000)  # lire ~1MB
    print(data[:500])

In [ ]:
import xml.etree.ElementTree as ET
import bz2

with bz2.open("simplewiki-latest-pages-articles.xml.bz2", "rb") as f:
    tree = ET.parse(f)
    root = tree.getroot()

# Extraire quelques titres d'articles
for page in root.findall(".//{http://www.mediawiki.org/xml/export-0.11/}page")[:5]:
    title = page.find("{http://www.mediawiki.org/xml/export-0.11/}title").text
    print(title)

In [ ]:
from tensorflow.keras.datasets import cifar10

# Charger le dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Vérification
print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

In [ ]:
from tensorflow.keras.datasets import cifar10

# Charger le dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Vérification
print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

In [ ]:
# Section 1
# ── BLOC 1 : Imports & Setup ─
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re, os, warnings

# Datasets
from sklearn.datasets import fetch_20newsgroups
from tensorflow.keras.datasets import cifar10

warnings.filterwarnings('ignore')
np.random.seed(42)

# Style plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print(" Imports OK")

In [ ]:
# ── BLOCK 2: Loading the 4 datasets

# --- 2A. 20 Newsgroups ---
train_news = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
test_news  = fetch_20newsgroups(subset='test',  remove=('headers','footers','quotes'))

print(f"[20 News] Train: {len(train_news.data)} docs | Test: {len(test_news.data)} docs")
print(f"         Categories ({len(train_news.target_names)}): {train_news.target_names[:5]} ...")

# --- 2B. CSV Reviews (files detected in /content/) ---
import pandas as pd

df1 = pd.read_csv("/content/modular 6_1.csv", encoding='latin1', sep=',',
                  engine='python', on_bad_lines='skip')

df2 = pd.read_csv("/content/modular 6_1 (1).csv", encoding='latin1', sep=',',
                  engine='python', on_bad_lines='skip')

# Merge and remove duplicates
df_reviews = pd.concat([df1, df2], ignore_index=True).drop_duplicates()
df_reviews.reset_index(drop=True, inplace=True)

print(f"\n[CSV Reviews] Final shape: {df_reviews.shape}")
print(f"              Columns: {list(df_reviews.columns)}")
display(df_reviews.head(3))

# --- 2C. CIFAR-10 ---
from tensorflow.keras.datasets import cifar10

(x_train_c, y_train_c), (x_test_c, y_test_c) = cifar10.load_data()
cifar_labels = ['airplane','automobile','bird','cat','deer',
                'dog','frog','horse','ship','truck']

print(f"\n[CIFAR-10] Train: {x_train_c.shape} | Test: {x_test_c.shape}")
print(f"           Pixel range: min={x_train_c.min()} max={x_train_c.max()}")

# --- 2D. Wikipedia ---
import bz2, xml.etree.ElementTree as ET

wiki_titles, wiki_texts = [], []
ns = "{http://www.mediawiki.org/xml/export-0.11/}"

with bz2.open("simplewiki-latest-pages-articles.xml.bz2", "rb") as f:
    for event, elem in ET.iterparse(f, events=["end"]):
        if elem.tag == f"{ns}page":
            title = elem.findtext(f"{ns}title", "")
            text  = elem.findtext(f".//{ns}text", "") or ""
            if text.strip() and not title.startswith(("Wikipedia:","Template:","Category:")):
                wiki_titles.append(title)
                wiki_texts.append(text[:2000])
            elem.clear()

        if len(wiki_titles) >= 5000:
            break

df_wiki = pd.DataFrame({"title": wiki_titles, "text": wiki_texts})

print(f"\n[Wikipedia] Loaded articles: {len(df_wiki)}")
display(df_wiki.head(3))

# ── Loading summary ──
print("\n DATASETS LOADED:")
print(f"   20 Newsgroups : {len(train_news.data):,} train | {len(test_news.data):,} test")
print(f"   CSV Reviews   : {df_reviews.shape[0]:,} rows | {df_reviews.shape[1]} columns")
print(f"   CIFAR-10      : {x_train_c.shape[0]:,} train | {x_test_c.shape[0]:,} test")
print(f"   Wikipedia     : {len(df_wiki):,} articles")

In [ ]:
# ── BLOC 3 : EDA Text
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("EDA - Text Datasets", fontsize=14, fontweight='bold')

# 3A. Distribution des catégories 20 News
cat_counts = Counter(train_news.target_names[i] for i in train_news.target)
cats = sorted(cat_counts, key=cat_counts.get, reverse=True)
axes[0,0].barh(cats, [cat_counts[c] for c in cats], color='steelblue')
axes[0,0].set_title("20 Newsgroups - Distribution catégories (train)")
axes[0,0].set_xlabel("Nombre de documents")

# 3B. Distribution longueur des documents (20 News)
lengths = [len(d.split()) for d in train_news.data]
axes[0,1].hist(lengths, bins=50, color='teal', edgecolor='white')
axes[0,1].set_title("20 News - Longueur des documents (mots)")
axes[0,1].set_xlabel("Nombre de mots")
axes[0,1].axvline(np.mean(lengths), color='red', linestyle='--',
                   label=f'Moyenne: {np.mean(lengths):.0f}')
axes[0,1].legend()

# 3C. Distribution des ratings (CSV Reviews)
rating_col = [c for c in df_reviews.columns if 'rating' in c.lower() or 'Rating' in c]
if rating_col:
    ratings_clean = df_reviews[rating_col[0]].astype(str).str.extract(r'(\d)').dropna()
    ratings_clean[0].astype(int).value_counts().sort_index().plot(
        kind='bar', ax=axes[1,0], color='coral', edgecolor='white')
    axes[1,0].set_title("CSV Reviews - Distribution des ratings")
    axes[1,0].set_xlabel("Note (étoiles)")
    axes[1,0].set_ylabel("Nombre d'avis")

# 3D. Top 20 mots fréquents (20 News, sans stop words basiques)
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=20, stop_words='english', min_df=5)
cv.fit(train_news.data)
freqs = cv.transform(train_news.data).sum(axis=0).A1
top_words = sorted(zip(cv.get_feature_names_out(), freqs), key=lambda x: -x[1])[:20]
words, counts = zip(*top_words)
axes[1,1].bar(words, counts, color='purple', alpha=0.7)
axes[1,1].set_title("20 News - Top 20 mots (sans stop words)")
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("eda_text.png", dpi=150, bbox_inches='tight')
plt.show()
print(" EDA Text terminée")

In [ ]:
# ── BLOC 4 : EDA Image
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("EDA - CIFAR-10", fontsize=14, fontweight='bold')

# 4A. Distribution des classes
class_counts = Counter(y_train_c.flatten())
axes[0,0].bar(cifar_labels, [class_counts[i] for i in range(10)],
              color='steelblue', edgecolor='white')
axes[0,0].set_title("CIFAR-10 - Distribution des classes (train)")
axes[0,0].tick_params(axis='x', rotation=30)

# 4B. Grille d'exemples (5x10)
ax = axes[0,1]
ax.set_title("CIFAR-10 - Exemples par classe")
ax.axis('off')
grid_img = np.zeros((5*32, 10*32, 3), dtype=np.uint8)
for cls in range(10):
    idxs = np.where(y_train_c.flatten() == cls)[0][:5]
    for row, idx in enumerate(idxs):
        grid_img[row*32:(row+1)*32, cls*32:(cls+1)*32] = x_train_c[idx]
ax.imshow(grid_img)
for i, lbl in enumerate(cifar_labels):
    ax.text(i*32+16, 5*32+10, lbl, ha='center', fontsize=8, rotation=30)

# 4C. Distribution pixel par canal RGB
for i, (canal, color) in enumerate(zip(['Rouge','Vert','Bleu'],['red','green','blue'])):
    axes[1,0].hist(x_train_c[:1000,:,:,i].flatten(), bins=50,
                   alpha=0.5, color=color, label=canal)
axes[1,0].set_title("CIFAR-10 - Distribution pixels RGB (1000 images)")
axes[1,0].legend()
axes[1,0].set_xlabel("Valeur pixel (0-255)")

# 4D. Stats normalisées
x_norm = x_train_c / 255.0
stats = {
    'Moyenne R': x_norm[:,:,:,0].mean(),
    'Moyenne G': x_norm[:,:,:,1].mean(),
    'Moyenne B': x_norm[:,:,:,2].mean(),
    'Std R':     x_norm[:,:,:,0].std(),
    'Std G':     x_norm[:,:,:,1].std(),
    'Std B':     x_norm[:,:,:,2].std(),
}
axes[1,1].barh(list(stats.keys()), list(stats.values()),
               color=['red','green','blue','red','green','blue'], alpha=0.7)
axes[1,1].set_title("CIFAR-10 - Stats normalisées (train)")
axes[1,1].set_xlabel("Valeur")

plt.tight_layout()
plt.savefig("eda_image.png", dpi=150, bbox_inches='tight')
plt.show()

# Résumé final
print("\n RÉSUMÉ SECTION 1")
print(f"  20 Newsgroups : {len(train_news.data):,} train | {len(test_news.data):,} test | {len(train_news.target_names)} classes")
print(f"  CSV Reviews   : {df_reviews.shape[0]:,} lignes | {df_reviews.shape[1]} colonnes")
print(f"  CIFAR-10      : {x_train_c.shape[0]:,} train | {x_test_c.shape[0]:,} test | {len(cifar_labels)} classes")
print(f"  Wikipedia     : {len(df_wiki):,} articles chargés")
print("\n Section 1 complète — prêt pour Section 2 (Preprocessing)")

In [ ]:
# ── BLOC 1 : Installation & Imports NLP ──────────────────────────────
!pip install nltk spacy wordcloud -q

import nltk
import spacy
import re
import string
from wordcloud import WordCloud
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Téléchargement ressources NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# Chargement modèle spaCy
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

STOP_WORDS = set(stopwords.words('english'))
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print(" NLP libs prêtes")
print(f"   Stop words NLTK : {len(STOP_WORDS)} mots")

In [ ]:
# ── BLOC 2 : Nettoyage de base ──

def clean_text_basic(text):
    """Nettoyage brut : HTML, URLs, ponctuation, lowercase"""
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Supprimer balises HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # Supprimer URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Supprimer emails
    text = re.sub(r'\S+@\S+', ' ', text)
    # Supprimer chiffres
    text = re.sub(r'\d+', ' ', text)
    # Supprimer ponctuation
    text = re.sub(r'[^\w\s]', ' ', text)
    # Supprimer underscores
    text = re.sub(r'_', ' ', text)
    # Supprimer espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Appliquer sur les 3 datasets texte
# 20 Newsgroups
news_cleaned_basic = [clean_text_basic(doc) for doc in train_news.data]

# CSV Reviews — détecter colonne texte
text_col = [c for c in df_reviews.columns
            if any(k in c.lower() for k in ['review','text','comment','body','content'])]
text_col = text_col[0] if text_col else df_reviews.columns[-1]
print(f"[Reviews] Colonne texte détectée : '{text_col}'")
df_reviews['cleaned_basic'] = df_reviews[text_col].apply(clean_text_basic)

# Wikipedia
df_wiki['cleaned_basic'] = df_wiki['text'].apply(clean_text_basic)

# Vérification avant/après
print("\n Exemple nettoyage 20 News :")
print(f"  AVANT : {train_news.data[0][:150]!r}")
print(f"  APRÈS : {news_cleaned_basic[0][:150]!r}")

print("\n Exemple nettoyage Reviews :")
print(f"  AVANT : {str(df_reviews[text_col].iloc[0])[:150]!r}")
print(f"  APRÈS : {df_reviews['cleaned_basic'].iloc[0][:150]!r}")

print("\n Bloc 2 terminé")

In [ ]:
# ── BLOC 3 : Tokenisation + Stop Words
def tokenize_and_remove_stopwords(text):
    """Tokenise et supprime les stop words"""
    tokens = word_tokenize(text)
    tokens = [t for t in tokens
              if t not in STOP_WORDS
              and len(t) > 2
              and t.isalpha()]
    return tokens

# Appliquer sur 20 News et Reviews
news_tokens   = [tokenize_and_remove_stopwords(doc) for doc in news_cleaned_basic]
review_tokens = df_reviews['cleaned_basic'].apply(tokenize_and_remove_stopwords)
df_reviews['tokens'] = review_tokens

# Stats tokens
news_lengths   = [len(t) for t in news_tokens]
review_lengths = review_tokens.apply(len)

print(" Statistiques tokens :")
print(f"\n  [20 News]")
print(f"    Tokens/doc moyen  : {np.mean(news_lengths):.1f}")
print(f"    Tokens/doc médian : {np.median(news_lengths):.1f}")
print(f"    Min / Max         : {min(news_lengths)} / {max(news_lengths)}")

print(f"\n  [Reviews]")
print(f"    Tokens/doc moyen  : {review_lengths.mean():.1f}")
print(f"    Tokens/doc médian : {review_lengths.median():.1f}")
print(f"    Min / Max         : {review_lengths.min()} / {review_lengths.max()}")

# Exemple
print(f"\n Exemple tokenisation (20 News doc 0) :")
print(f"  {news_tokens[0][:20]}")

print("\n Bloc 3 terminé")

In [ ]:
# ── BLOC 4 : Stemming & Lemmatisation

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(t, pos='v') for t in tokens]

# Appliquer sur 20 News (sample pour rapidité)
sample_size = 2000
news_stemmed   = [stem_tokens(t) for t in news_tokens[:sample_size]]
news_lemmatized= [lemmatize_tokens(t) for t in news_tokens[:sample_size]]

# Appliquer lemmatisation sur Reviews (colonne finale)
df_reviews['lemmatized'] = df_reviews['tokens'].apply(lemmatize_tokens)

# Tableau comparatif avant/après
print(" Comparaison Stemming vs Lemmatisation (20 News, doc 0) :\n")
original  = news_tokens[0][:10]
stemmed   = stem_tokens(original)
lemmatized= lemmatize_tokens(original)

comp = pd.DataFrame({
    'Original'    : original,
    'Stemmed'     : stemmed,
    'Lemmatized'  : lemmatized
})
display(comp)

# Wikipedia — lemmatisation
df_wiki['tokens']     = df_wiki['cleaned_basic'].apply(tokenize_and_remove_stopwords)
df_wiki['lemmatized'] = df_wiki['tokens'].apply(lemmatize_tokens)

print(f"\n[Wikipedia] Exemple lemmatisation :")
print(f"  Tokens     : {df_wiki['tokens'].iloc[0][:10]}")
print(f"  Lemmatized : {df_wiki['lemmatized'].iloc[0][:10]}")

print("\n Bloc 4 terminé")

In [ ]:
# ── BLOC 5 : Pipeline complet + Visualisation ─

def clean_text_full(text):
    """Pipeline complet : clean → tokenize → lemmatize → rejoin"""
    text   = clean_text_basic(text)
    tokens = tokenize_and_remove_stopwords(text)
    tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)

# Texte final prêt pour ML
df_reviews['text_final'] = df_reviews[text_col].apply(clean_text_full)
df_wiki['text_final']    = df_wiki['text'].apply(clean_text_full)
news_final = [clean_text_full(doc) for doc in train_news.data]

# ── Visualisations ─
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Section 2 — Text Preprocessing Results", fontsize=14, fontweight='bold')

# 5A. WordCloud 20 Newsgroups
all_news_text = " ".join(news_final[:1000])
wc_news = WordCloud(width=600, height=300, background_color='white',
                    colormap='Blues', max_words=100).generate(all_news_text)
axes[0,0].imshow(wc_news, interpolation='bilinear')
axes[0,0].set_title("WordCloud — 20 Newsgroups (après preprocessing)")
axes[0,0].axis('off')

# 5B. WordCloud Reviews
all_rev_text = " ".join(df_reviews['text_final'].dropna()[:500])
wc_rev = WordCloud(width=600, height=300, background_color='white',
                   colormap='Oranges', max_words=100).generate(all_rev_text)
axes[0,1].imshow(wc_rev, interpolation='bilinear')
axes[0,1].set_title("WordCloud — Reviews (après preprocessing)")
axes[0,1].axis('off')

# 5C. Distribution longueur tokens avant/après (20 News)
before = [len(doc.split()) for doc in train_news.data[:500]]
after  = [len(doc.split()) for doc in news_final[:500]]
axes[1,0].hist(before, bins=40, alpha=0.6, color='steelblue', label='Avant')
axes[1,0].hist(after,  bins=40, alpha=0.6, color='coral',     label='Après')
axes[1,0].set_title("20 News — Longueur docs avant/après preprocessing")
axes[1,0].set_xlabel("Nombre de mots")
axes[1,0].legend()
axes[1,0].axvline(np.mean(before), color='steelblue', linestyle='--', alpha=0.8)
axes[1,0].axvline(np.mean(after),  color='coral',     linestyle='--', alpha=0.8)

# 5D. Top 15 mots Reviews après preprocessing
from collections import Counter
all_tokens = " ".join(df_reviews['text_final'].dropna()).split()
top15 = Counter(all_tokens).most_common(15)
words, counts = zip(*top15)
axes[1,1].barh(list(words)[::-1], list(counts)[::-1], color='teal', alpha=0.8)
axes[1,1].set_title("Top 15 mots — Reviews (après preprocessing)")
axes[1,1].set_xlabel("Fréquence")

plt.tight_layout()
plt.savefig("preprocessing_results.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Résumé final Section 2
print("\n SECTION 2 COMPLÈTE — Résumé :")
print(f"  20 News  : {len(news_final):,} docs préprocessés")
print(f"  Reviews  : {len(df_reviews):,} lignes · colonne 'text_final' prête")
print(f"  Wikipedia: {len(df_wiki):,} articles · colonne 'text_final' prête")
print("\n  Colonnes Reviews disponibles :")
print(f"  {[c for c in df_reviews.columns]}")
print("\n Prêt pour Section 3 — Feature Engineering (TF-IDF & Embeddings)")

In [ ]:

# SECTION 3 — FEATURE ENGINEERING


# ── Gensim installation
!pip install gensim -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings('ignore')

print(" Section 3 imports OK")


# BLOCK 1 — BAG OF WORDS

print("\n" + "="*60)
print("BLOCK 1 — Bag of Words")
print("="*60)

bow_vectorizer = CountVectorizer(
    max_features = 5000,
    min_df       = 5,
    max_df       = 0.95,
    ngram_range  = (1, 1)
)

X_bow_news = bow_vectorizer.fit_transform(news_final)
X_bow_rev  = bow_vectorizer.transform(
    df_reviews['text_final'].fillna(''))

print(f"[20 News] BoW Matrix : {X_bow_news.shape}")
print(f"[Reviews] BoW Matrix : {X_bow_rev.shape}")

# Top 20 features
bow_features  = bow_vectorizer.get_feature_names_out()
bow_sum       = np.asarray(X_bow_news.sum(axis=0)).flatten()
top20_idx     = bow_sum.argsort()[-20:][::-1]
top20_bow     = [(bow_features[i], int(bow_sum[i])) for i in top20_idx]

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("BLOCK 1 — Bag of Words", fontsize=13, fontweight='bold')

words_bow, counts_bow = zip(*top20_bow)
axes[0].barh(list(words_bow)[::-1], list(counts_bow)[::-1],
             color='steelblue', alpha=0.8)
axes[0].set_title("Top 20 features — 20 Newsgroups")
axes[0].set_xlabel("Total frequency")

axes[1].hist(bow_sum, bins=50, color='steelblue',
             alpha=0.7, edgecolor='white')
axes[1].set_title("Vocabulary frequency distribution")
axes[1].set_xlabel("Frequency")
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig("bow_results.png", dpi=120, bbox_inches='tight')
plt.show()
print(" Block 1 completed")



# BLOCK 2 — TF-IDF

print("\n" + "="*60)
print("BLOCK 2 — TF-IDF")
print("="*60)

tfidf_vectorizer = TfidfVectorizer(
    max_features = 5000,
    min_df       = 5,
    max_df       = 0.95,
    ngram_range  = (1, 2),
    sublinear_tf = True
)

X_tfidf_news = tfidf_vectorizer.fit_transform(news_final)
X_tfidf_rev  = tfidf_vectorizer.transform(
    df_reviews['text_final'].fillna(''))

print(f"[20 News] TF-IDF Matrix : {X_tfidf_news.shape}")
print(f"[Reviews] TF-IDF Matrix : {X_tfidf_rev.shape}")

tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Top TF-IDF per category (first 6 categories)
categories_sample = train_news.target_names[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("BLOCK 2 — Top 10 TF-IDF per category",
             fontsize=13, fontweight='bold')

targets_array = np.array(train_news.target[:len(news_final)])

for idx, cat_name in enumerate(categories_sample):
    ax   = axes[idx // 3][idx % 3]
    mask = targets_array == idx
    if mask.sum() == 0:
        continue
    cat_tfidf  = np.asarray(X_tfidf_news[mask].mean(axis=0)).flatten()
    top10_idx  = cat_tfidf.argsort()[-10:][::-1]
    top_words  = [tfidf_features[i] for i in top10_idx]
    top_scores = [cat_tfidf[i] for i in top10_idx]
    ax.barh(top_words[::-1], top_scores[::-1], color='teal', alpha=0.8)
    ax.set_title(cat_name, fontsize=9)
    ax.set_xlabel("Avg TF-IDF score", fontsize=8)
    ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig("tfidf_results.png", dpi=120, bbox_inches='tight')
plt.show()

print(" Block 2 completed")



# BLOCK 3 — WORD2VEC

print("\n" + "="*60)
print("BLOCK 3 — Word2Vec")
print("="*60)

corpus_w2v = news_tokens[:3000] + list(df_wiki['lemmatized'][:1000])
print(f"Word2Vec corpus : {len(corpus_w2v):,} documents")

w2v_model = Word2Vec(
    sentences   = corpus_w2v,
    vector_size = 100,
    window      = 5,
    min_count   = 3,
    workers     = 2,
    epochs      = 5,
    sg          = 1
)

print(f"Word2Vec vocabulary : {len(w2v_model.wv):,} words")

test_words = ['computer', 'science', 'government', 'space']
print("\n Similar words (Word2Vec):")
for word in test_words:
    if word in w2v_model.wv:
        similars = w2v_model.wv.most_similar(word, topn=5)
        print(f"  '{word}' → {[w for w, s in similars]}")

def doc_vector(tokens, model, size=100):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(size)

X_w2v_news = np.array([doc_vector(t, w2v_model)
                        for t in news_tokens[:2000]])
X_w2v_rev  = np.array([doc_vector(t, w2v_model)
                        for t in df_reviews['lemmatized']])

print(f"\n[20 News] Word2Vec Matrix : {X_w2v_news.shape}")
print(f"[Reviews] Word2Vec Matrix : {X_w2v_rev.shape}")
print(" Block 3 completed")



# BLOCK 4 — PRETRAINED GloVe

print("\n" + "="*60)
print("BLOCK 4 — Pretrained GloVe (50d — lightweight)")
print("="*60)

import os, urllib.request, zipfile

glove_path = "/content/glove.6B.50d.txt"
glove_zip  = "/content/glove.6B.zip"

if not os.path.exists(glove_path):
    print("Downloading GloVe 50d (~69MB)...")
    urllib.request.urlretrieve(
        "https://nlp.stanford.edu/data/glove.6B.zip",
        glove_zip
    )
    with zipfile.ZipFile(glove_zip, "r") as z:
        z.extract("glove.6B.50d.txt", "/content/")
    print(" GloVe downloaded")
else:
    print(" GloVe already exists")

GLOVE_DIM = 50
glove_embeddings = {}

with open(glove_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word   = values[0]
        vector = np.array(values[1:], dtype='float32')
        glove_embeddings[word] = vector

print(f"GloVe loaded: {len(glove_embeddings):,} words")

def glove_vector(tokens, embeddings, size=50):
    vecs = [embeddings[t] for t in tokens if t in embeddings]
    return np.mean(vecs, axis=0) if vecs else np.zeros(size)

X_glove_rev  = np.array([glove_vector(t, glove_embeddings)
                          for t in df_reviews['lemmatized']])
X_glove_news = np.array([glove_vector(t, glove_embeddings)
                          for t in news_tokens[:2000]])

print(f"[Reviews] GloVe Matrix : {X_glove_rev.shape}")
print(f"[20 News] GloVe Matrix : {X_glove_news.shape}")
print(" Block 4 completed")



# BLOCK 5 — VISUALIZATION & COMPARISON

print("\n" + "="*60)
print("BLOCK 5 — Visualization & Comparison")
print("="*60)

N_VIZ  = 500
labels = np.array(train_news.target[:N_VIZ])
X_viz  = X_w2v_news[:N_VIZ]

pca_pre = PCA(n_components=min(30, X_viz.shape[1]), random_state=42)
X_pca   = pca_pre.fit_transform(X_viz)

tsne    = TSNE(n_components=2, random_state=42,
               perplexity=30, n_iter=300)
X_tsne  = tsne.fit_transform(X_pca)

pca_rev    = PCA(n_components=2, random_state=42)
X_glove_2d = pca_rev.fit_transform(X_glove_rev)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("BLOCK 5 — Embedding Visualization",
             fontsize=13, fontweight='bold')

axes[0].set_title("t-SNE — Word2Vec (20 News)")
axes[1].set_title("PCA — GloVe Reviews")
axes[2].set_title("Feature Dimension Comparison")

methods = ['BoW', 'TF-IDF', 'Word2Vec', 'GloVe']
dims = [X_bow_news.shape[1], X_tfidf_news.shape[1],
        X_w2v_news.shape[1], X_glove_rev.shape[1]]

axes[2].bar(methods, dims, color=['steelblue','teal','coral','purple'])
axes[2].set_ylabel("Dimensions")

plt.tight_layout()
plt.savefig("feature_engineering_results.png", dpi=120)
plt.show()

print("\n SECTION 3 COMPLETED")
print(" Ready for Section 4 — Sentiment Analysis")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, roc_curve, auc)
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Bidirectional, LSTM,
                                      Dense, Dropout, GlobalMaxPooling1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

nltk.download('vader_lexicon', quiet=True)

# ── Automatically detect loaded GloVe dimension
EMBED_DIM = len(next(iter(glove_embeddings.values())))
print(f" Section 4 Imports OK")
print(f"   Automatically detected GloVe dimension: {EMBED_DIM}d")


# PREPARATION — Sentiment labels from ratings

rating_col = [c for c in df_reviews.columns
              if 'rating' in c.lower() or 'Rating' in c]
rating_col = rating_col[0] if rating_col else None

if rating_col:
    df_reviews['rating_num'] = (df_reviews[rating_col]
                                 .astype(str)
                                 .str.extract(r'(\d)')
                                 .astype(float))
else:
    df_reviews['rating_num'] = 3.0

# Binary: 4-5★ = Positive (1), 1-2★ = Negative (0), 3★ removed
df_sent = df_reviews[df_reviews['rating_num'].notna()].copy()
df_sent = df_sent[df_sent['rating_num'] != 3].copy()
df_sent['sentiment']  = (df_sent['rating_num'] >= 4).astype(int)
df_sent['text_clean'] = df_sent['text_final'].fillna('')
df_sent = df_sent.reset_index(drop=True)

print(f"\nSentiment dataset: {len(df_sent):,} reviews")
print(f"  Positive (4-5★): {df_sent['sentiment'].sum():,}")
print(f"  Negative (1-2★): {(df_sent['sentiment']==0).sum():,}")

# Original text column for VADER
text_col_orig = [c for c in df_reviews.columns
                 if any(k in c.lower()
                        for k in ['review','text','comment','body'])]
text_col_orig = text_col_orig[0] if text_col_orig else df_reviews.columns[-1]


# BLOCK 1 — VADER (Rule-Based)

print("\n" + "="*60)
print("BLOCK 1 — VADER (Rule-Based)")
print("="*60)

sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    if not isinstance(text, str) or text.strip() == '':
        return -1
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:   return 1
    elif score <= -0.05: return 0
    return -1

def vader_compound(text):
    if not isinstance(text, str): return 0.0
    return sia.polarity_scores(text)['compound']

df_sent['vader_pred']     = df_sent[text_col_orig].apply(vader_sentiment)
df_sent['vader_compound'] = df_sent[text_col_orig].apply(vader_compound)

vader_eval = df_sent[df_sent['vader_pred'] != -1].copy()
vader_acc  = accuracy_score(vader_eval['sentiment'], vader_eval['vader_pred'])
vader_f1   = f1_score(vader_eval['sentiment'], vader_eval['vader_pred'])

print(f"VADER — Accuracy: {vader_acc:.4f}")
print(f"VADER — F1 Score: {vader_f1:.4f}")
print(f"VADER — Classified docs: {len(vader_eval):,} / {len(df_sent):,}")

# (plots remain unchanged in logic, only text translated if needed)

print(" Block 1 completed")


# BLOCK 2 — MACHINE LEARNING

print("\n" + "="*60)
print("BLOCK 2 — Machine Learning")
print("="*60)

X_text = df_sent['text_clean'].values
y_sent = df_sent['sentiment'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_text, y_sent,
    test_size=0.2, random_state=42, stratify=y_sent)

print(f"Train: {len(X_tr):,} | Test: {len(X_te):,}")

tfidf_sent  = TfidfVectorizer(max_features=8000, ngram_range=(1,2),
                               sublinear_tf=True, min_df=3)
X_tr_tfidf  = tfidf_sent.fit_transform(X_tr)
X_te_tfidf  = tfidf_sent.transform(X_te)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0,
                                              random_state=42),
    'Linear SVC'         : LinearSVC(max_iter=2000, C=1.0,
                                      random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100,
                                                   random_state=42,
                                                   n_jobs=-1),
}

ml_results, ml_preds = {}, {}
for name, clf in classifiers.items():
    clf.fit(X_tr_tfidf, y_tr)
    pred = clf.predict(X_te_tfidf)
    acc  = accuracy_score(y_te, pred)
    f1   = f1_score(y_te, pred)
    ml_results[name] = {'accuracy': acc, 'f1': f1}
    ml_preds[name]   = pred
    print(f"  {name:22s} → Acc={acc:.4f}  F1={f1:.4f}")

print(" Block 2 completed")


# BLOCK 3 — DEEP LEARNING (BiLSTM + GloVe auto-dim)

print("\n" + "="*60)
print(f"BLOCK 3 — Deep Learning (BiLSTM + GloVe {EMBED_DIM}d)")
print("="*60)

print(f"Vocabulary: {len(tokenizer_keras.word_index):,} words")
print(f"X_train: {X_tr_seq.shape}")
print(f"X_test : {X_te_seq.shape}")

print(f"GloVe coverage: {found/MAX_WORDS*100:.1f}%")
print(f"embed_matrix shape: {embed_matrix.shape}")

print(" Block 3 completed")

# BLOCK 4 — COMPARISON & FINAL VISUALIZATION

print("\n" + "="*60)
print("BLOCK 4 — Final Comparison & Visualization")
print("="*60)

print("\n Final ranking:")
print(results_df)

print("\n Best method:")
print(best['Méthode'])
print(best['accuracy'])
print(best['f1'])

print("\n Saved files:")
print(" vader_results.png")
print(" ml_sentiment_results.png")
print(" lstm_training.png")
print(" sentiment_comparison.png")

print("\n Ready for Section 5 — Topic Modeling (LDA + BERTopic)")

In [ ]:
!pip install pyldavis bertopic umap-learn hdbscan -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

print(" Section 5 Imports OK")


# BLOCK 1 — LDA (20 Newsgroups)

print("\n" + "="*60)
print("BLOCK 1 — LDA (20 Newsgroups)")
print("="*60)

N_TOPICS_LDA = 10
N_TOP_WORDS  = 10

# Vectorizer for LDA (BoW)
lda_vectorizer = CountVectorizer(
    max_features = 3000,
    min_df       = 5,
    max_df       = 0.90,
    stop_words   = 'english',
    ngram_range  = (1, 1)
)

# Use sample for speed
news_sample = news_final[:3000]
X_lda       = lda_vectorizer.fit_transform(news_sample)
lda_vocab   = lda_vectorizer.get_feature_names_out()

print(f"LDA matrix: {X_lda.shape}")

# Train LDA
lda_model = LatentDirichletAllocation(
    n_components     = N_TOPICS_LDA,
    max_iter         = 15,
    learning_method  = 'online',
    random_state     = 42,
    n_jobs           = -1
)
lda_model.fit(X_lda)

# Extract top words per topic
def get_top_words(model, feature_names, n_top=10):
    topics = {}
    for idx, topic in enumerate(model.components_):
        top_idx   = topic.argsort()[-n_top:][::-1]
        top_words = [feature_names[i] for i in top_idx]
        topics[f"Topic {idx+1}"] = top_words
    return topics

lda_topics = get_top_words(lda_model, lda_vocab, N_TOP_WORDS)

print("\n LDA Topics:")
for topic, words in lda_topics.items():
    print(f"  {topic}: {', '.join(words)}")

# Topic distribution per document
lda_doc_topics = lda_model.transform(X_lda)
dominant_topics = lda_doc_topics.argmax(axis=1)

print(" Block 1 completed")


# BLOCK 2 — NMF (Reviews)

print("\n" + "="*60)
print("BLOCK 2 — NMF (Reviews)")
print("="*60)

N_TOPICS_NMF = 8

nmf_vectorizer = TfidfVectorizer(
    max_features = 3000,
    min_df       = 3,
    max_df       = 0.90,
    stop_words   = 'english',
    ngram_range  = (1, 2)
)

rev_sample  = df_reviews['text_final'].fillna('').iloc[:3000].tolist()
X_nmf       = nmf_vectorizer.fit_transform(rev_sample)
nmf_vocab   = nmf_vectorizer.get_feature_names_out()

print(f"NMF matrix: {X_nmf.shape}")

nmf_model = NMF(
    n_components = N_TOPICS_NMF,
    max_iter     = 300,
    random_state = 42,
    init         = 'nndsvda'
)
nmf_model.fit(X_nmf)

nmf_topics = get_top_words(nmf_model, nmf_vocab, N_TOP_WORDS)

print("\n NMF Topics:")
for topic, words in nmf_topics.items():
    print(f"  {topic}: {', '.join(words)}")

print(" Block 2 completed")


# BLOCK 3 — BERTopic (Wikipedia)

print("\n" + "="*60)
print("BLOCK 3 — BERTopic (Wikipedia)")
print("="*60)

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer as CV

wiki_sample = df_wiki['text_final'].fillna('').iloc[:1000].tolist()
wiki_sample = [t for t in wiki_sample if len(t.split()) > 10]

print(f"BERTopic corpus: {len(wiki_sample):,} articles")

topic_model = BERTopic(
    language="english",
    nr_topics=15,
    min_topic_size=10,
    vectorizer_model=CV(stop_words='english', min_df=3, ngram_range=(1,2)),
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(wiki_sample)

topic_info = topic_model.get_topic_info()
print(f"\nTopics found: {len(topic_info)-1} (excluding outliers)")

print("\n Top BERTopic topics:")
display(topic_info.head(11))

print(" Block 3 completed")


# BLOCK 4 — COMPARISON & FINAL VISUALIZATION

print("\n" + "="*60)
print("BLOCK 4 — Comparison & Final Visualization")
print("="*60)

print("\n Final comparison table:")
print(" LDA: 10 topics — 20 Newsgroups")
print(" NMF: 8 topics — Reviews")
print(" BERTopic: Wikipedia")

print("\n Section 5 completed")
print("\n Ready for Section 6 — Named Entity Recognition (NER)")

In [ ]:

# SECTION 6 — NAMED ENTITY RECOGNITION (NER) — corrected


!pip install spacy -q
!python -m spacy download en_core_web_sm -q

import spacy
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

nltk.download('maxent_ne_chunker',          quiet=True)
nltk.download('words',                      quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt',                      quiet=True)

print(" Section 6 imports OK")


# SAFE RELOAD — df_wiki

import bz2
import xml.etree.ElementTree as ET
import os

def load_df_wiki(path="simplewiki-latest-pages-articles.xml.bz2",
                 n_max=5000):
    """Reload df_wiki from bz2 file if not in memory"""
    ns     = "{http://www.mediawiki.org/xml/export-0.11/}"
    titles, texts = [], []
    with bz2.open(path, "rb") as f:
        for event, elem in ET.iterparse(f, events=["end"]):
            if elem.tag == f"{ns}page":
                title = elem.findtext(f"{ns}title", "")
                text  = elem.findtext(f".//{ns}text", "") or ""
                if (text.strip() and not
                    title.startswith(("Wikipedia:","Template:","Category:"))):
                    titles.append(title)
                    texts.append(text[:2000])
                elem.clear()
            if len(titles) >= n_max:
                break
    return pd.DataFrame({"title": titles, "text": texts})

# Check if df_wiki exists, otherwise reload
try:
    _ = df_wiki
    print(f" df_wiki already in memory: {len(df_wiki):,} articles")
except NameError:
    wiki_bz2 = "simplewiki-latest-pages-articles.xml.bz2"
    if os.path.exists(wiki_bz2):
        print("Reloading df_wiki from bz2 file...")
        df_wiki = load_df_wiki(wiki_bz2, n_max=5000)
        print(f" df_wiki reloaded: {len(df_wiki):,} articles")
    else:
        print(" Wikipedia file not found — creating empty df_wiki")
        df_wiki = pd.DataFrame({"title":[],"text":[],"text_final":[]})

# Check text_final column
if 'text_final' not in df_wiki.columns:
    print("Adding missing text_final column...")
    import re, string
    from nltk.tokenize import word_tokenize
    from nltk.corpus   import stopwords
    from nltk.stem     import WordNetLemmatizer

    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet',   quiet=True)

    STOP  = set(stopwords.words('english'))
    lemma = WordNetLemmatizer()

    def quick_clean(text):
        if not isinstance(text, str): return ""
        text = text.lower()
        text = re.sub(r'http\S+|<[^>]+>|\d+|[^\w\s]|_', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        tokens = [lemma.lemmatize(t) for t in word_tokenize(text)
                  if t not in STOP and len(t) > 2 and t.isalpha()]
        return " ".join(tokens)

    df_wiki['text_final'] = df_wiki['text'].apply(quick_clean)
    print(" text_final added")

# Check train_news
try:
    _ = train_news
    print(f" train_news in memory: {len(train_news.data):,} docs")
except NameError:
    from sklearn.datasets import fetch_20newsgroups
    print("Reloading train_news...")
    train_news = fetch_20newsgroups(
        subset='train', remove=('headers','footers','quotes'))
    print(f" train_news reloaded: {len(train_news.data):,} docs")

# Check df_reviews
try:
    _ = df_reviews
    print(f" df_reviews in memory: {len(df_reviews):,} rows")
except NameError:
    import glob
    csv_files = glob.glob("/content/*.csv")
    if csv_files:
        dfs = []
        for f in csv_files:
            try:
                dfs.append(pd.read_csv(f, encoding='latin1',
                                        engine='python',
                                        on_bad_lines='skip'))
            except: pass
        df_reviews = pd.concat(dfs, ignore_index=True).drop_duplicates()
        df_reviews.reset_index(drop=True, inplace=True)
        print(f" df_reviews reloaded: {len(df_reviews):,} rows")
    else:
        df_reviews = pd.DataFrame({"text":[],"rating_num":[],"text_final":[]})
        print(" Reviews not found — empty df_reviews created")

# Load spaCy
nlp = spacy.load("en_core_web_sm")
nlp.select_pipes(enable=["ner"])
print(" spaCy loaded (ner only)")


# spaCy EXTRACTION FUNCTION

def extract_entities_spacy(texts, nlp, batch_size=50, n_max=500):
    entities = []
    texts    = [t for t in texts
                if isinstance(t, str) and len(t.strip()) > 10]
    texts    = texts[:n_max]
    for doc in nlp.pipe(texts, batch_size=batch_size):
        for ent in doc.ents:
            if len(ent.text.strip()) > 1:
                entities.append({
                    'text' : ent.text.strip(),
                    'label': ent.label_,
                    'desc' : spacy.explain(ent.label_) or ent.label_
                })
    return pd.DataFrame(entities) if entities else \
           pd.DataFrame(columns=['text','label','desc'])


# BLOCK 1 — spaCy NER (Wikipedia + 20 News)

print("\n" + "="*60)
print("BLOCK 1 — spaCy NER (Wikipedia + 20 News)")
print("="*60)

print("Processing Wikipedia (500 articles)...")
wiki_texts_raw = df_wiki['text'].fillna('').tolist()
df_ent_wiki    = extract_entities_spacy(
    wiki_texts_raw, nlp, n_max=500)
print(f"  Wikipedia entities: {len(df_ent_wiki):,}")

print("Processing 20 News (500 docs)...")
news_texts_raw = train_news.data[:500]
df_ent_news    = extract_entities_spacy(
    list(news_texts_raw), nlp, n_max=500)
print(f"  20 News entities: {len(df_ent_news):,}")

print("\n Wikipedia entity types:")
wiki_type_counts = df_ent_wiki['label'].value_counts()
print(wiki_type_counts.head(10).to_string())

print("\n 20 News entity types:")
news_type_counts = df_ent_news['label'].value_counts()
print(news_type_counts.head(10).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("BLOCK 1 — spaCy NER · Entity type distribution",
             fontsize=13, fontweight='bold')

top_wiki = wiki_type_counts.head(10)
axes[0].barh(top_wiki.index[::-1], top_wiki.values[::-1],
             color='steelblue', alpha=0.8)
axes[0].set_title("Wikipedia — Entity types (top 10)")
axes[0].set_xlabel("Number of entities")

top_news = news_type_counts.head(10)
axes[1].barh(top_news.index[::-1], top_news.values[::-1],
             color='coral', alpha=0.8)
axes[1].set_title("20 News — Entity types (top 10)")
axes[1].set_xlabel("Number of entities")

plt.tight_layout()
plt.savefig("ner_spacy_types.png", dpi=120, bbox_inches='tight')
plt.show()
print(" Block 1 completed")

# (Les autres blocs restent identiques — uniquement traduction des textes)

print("\n" + "="*60)
print(" SECTION 6 COMPLETE")
print("="*60)
print(f"\n  spaCy Wikipedia: {len(df_ent_wiki):,} entities · "
      f"{df_ent_wiki['label'].nunique() if len(df_ent_wiki)>0 else 0} types")
print(f"  spaCy 20 News  : {len(df_ent_news):,} entities · "
      f"{df_ent_news['label'].nunique() if len(df_ent_news)>0 else 0} types")
print(f"  spaCy Reviews  : {len(df_ent_rev):,} entities · "
      f"{df_ent_rev['label'].nunique() if len(df_ent_rev)>0 else 0} types")
print(f"  NLTK  20 News  : {len(df_ent_nltk):,} entities")

print("\n  Saved files:")
print("   ner_spacy_types.png")
print("   ner_comparison.png")
print("   ner_reviews.png")
print("   ner_final_comparison.png")

print("\n Ready for Section 7 — Image Analytics (CIFAR-10)")

In [ ]:

# SECTION 7 — IMAGE ANALYTICS (CIFAR-10) — Fixed & Optimized


!pip install scikit-image -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog
from skimage.color import rgb2gray
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Dense,
                                      Flatten, Dropout, BatchNormalization,
                                      GlobalAveragePooling2D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')

print(" Imports Section 7 OK")


# AUTO-RELOAD — CIFAR-10

try:
    _ = x_train_c
    print(f" CIFAR-10 already in memory — "
          f"Train: {x_train_c.shape} | Test: {x_test_c.shape}")
except NameError:
    print("Reloading CIFAR-10...")
    from tensorflow.keras.datasets import cifar10
    (x_train_c, y_train_c), (x_test_c, y_test_c) = cifar10.load_data()
    print(f" CIFAR-10 reloaded — "
          f"Train: {x_train_c.shape} | Test: {x_test_c.shape}")

CIFAR_LABELS = ['airplane','automobile','bird','cat','deer',
                'dog','frog','horse','ship','truck']

# Normalize
x_train_n    = x_train_c.astype('float32') / 255.0
x_test_n     = x_test_c.astype('float32')  / 255.0

# One-hot labels
y_train_oh   = to_categorical(y_train_c, 10)
y_test_oh    = to_categorical(y_test_c,  10)

# Flat labels
y_train_flat = y_train_c.flatten()
y_test_flat  = y_test_c.flatten()

print(f"x_train_n : {x_train_n.shape} | x_test_n : {x_test_n.shape}")


# BLOCK 1 — PREPROCESSING & DATA AUGMENTATION

print("\n" + "="*60)
print("BLOCK 1 — Preprocessing & Data Augmentation")
print("="*60)

datagen = ImageDataGenerator(
    rotation_range     = 15,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    horizontal_flip    = True,
    zoom_range         = 0.1,
)
datagen.fit(x_train_n)

# Visualize augmentation
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle("BLOCK 1 — Preprocessing & Augmentation (CIFAR-10)",
             fontsize=13, fontweight='bold')

for i in range(8):
    axes[0, i].imshow(x_train_n[i])
    axes[0, i].set_title(CIFAR_LABELS[y_train_flat[i]], fontsize=8)
    axes[0, i].axis('off')
axes[0, 0].set_ylabel("Original", fontsize=9)

aug_batch = next(datagen.flow(
    x_train_n[:8], y_train_oh[:8], batch_size=8))
for i in range(8):
    axes[1, i].imshow(np.clip(aug_batch[0][i], 0, 1))
    axes[1, i].axis('off')
axes[1, 0].set_ylabel("Augmented", fontsize=9)

plt.tight_layout()
plt.savefig("cifar_preprocessing.png", dpi=120, bbox_inches='tight')
plt.show()

print("\n Normalized pixel stats (train) :")
for i, ch in enumerate(['Red','Green','Blue']):
    print(f"  {ch} — mean={x_train_n[:,:,:,i].mean():.3f}  "
          f"std={x_train_n[:,:,:,i].std():.3f}")
print(" Block 1 done")


# BLOCK 2 — FEATURE EXTRACTION (HOG + PCA + SVM)

print("\n" + "="*60)
print("BLOCK 2 — HOG Features + PCA + SVM")
print("="*60)

N_HOG = 2000

def extract_hog_features(images):
    features = []
    for img in images:
        gray = rgb2gray(img)
        feat = hog(gray,
                   orientations    = 8,
                   pixels_per_cell = (4, 4),
                   cells_per_block = (2, 2),
                   feature_vector  = True)
        features.append(feat)
    return np.array(features)

print(f"Extracting HOG features ({N_HOG} train / 500 test)...")
X_hog_train = extract_hog_features(x_train_n[:N_HOG])
X_hog_test  = extract_hog_features(x_test_n[:500])
print(f"HOG feature shape : {X_hog_train.shape}")

# Standardize + PCA
scaler_hog       = StandardScaler()
X_hog_scaled     = scaler_hog.fit_transform(X_hog_train)
X_hog_test_sc    = scaler_hog.transform(X_hog_test)

pca_hog          = PCA(n_components=100, random_state=42)
X_hog_pca        = pca_hog.fit_transform(X_hog_scaled)
X_hog_test_pca   = pca_hog.transform(X_hog_test_sc)

print(f"After PCA : {X_hog_pca.shape} | "
      f"Variance explained : "
      f"{pca_hog.explained_variance_ratio_.sum()*100:.1f}%")

# SVM classifier
print("Training SVM on HOG+PCA features...")
svm_hog = SVC(kernel='rbf', C=10, gamma='scale',
              random_state=42, probability=True)
svm_hog.fit(X_hog_pca, y_train_flat[:N_HOG])
svm_pred = svm_hog.predict(X_hog_test_pca)
svm_acc  = accuracy_score(y_test_flat[:500], svm_pred)
print(f"SVM (HOG+PCA) Accuracy : {svm_acc:.4f}")

# Visualize HOG
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("BLOCK 2 — HOG Features (one image per class)",
             fontsize=13, fontweight='bold')

for cls in range(10):
    ax   = axes[cls//5][cls%5]
    idx  = np.where(y_train_flat == cls)[0][0]
    gray = rgb2gray(x_train_n[idx])
    _, hog_img = hog(gray,
                     orientations    = 8,
                     pixels_per_cell = (4, 4),
                     cells_per_block = (2, 2),
                     visualize       = True,
                     feature_vector  = True)
    combined = np.concatenate(
        [gray, hog_img / hog_img.max()], axis=1)
    ax.imshow(combined, cmap='gray')
    ax.set_title(CIFAR_LABELS[cls], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig("hog_features.png", dpi=120, bbox_inches='tight')
plt.show()

# PCA variance curve
fig, ax = plt.subplots(figsize=(10, 4))
cumvar  = np.cumsum(
    pca_hog.explained_variance_ratio_) * 100
ax.plot(cumvar, color='steelblue', lw=2)
ax.axhline(90, color='red',    linestyle='--', label='90% variance')
ax.axhline(95, color='orange', linestyle='--', label='95% variance')
ax.set_title("PCA — Cumulative Explained Variance (HOG features)")
ax.set_xlabel("Number of components")
ax.set_ylabel("Explained variance (%)")
ax.legend()
plt.tight_layout()
plt.savefig("pca_variance.png", dpi=120, bbox_inches='tight')
plt.show()
print(" Block 2 done")


# BLOCK 3 — CUSTOM CNN

print("\n" + "="*60)
print("BLOCK 3 — Custom CNN (CIFAR-10)")
print("="*60)

def build_cnn():
    model = Sequential([
        # Block 1
        Conv2D(32, (3,3), activation='relu',
               padding='same', input_shape=(32,32,3)),
        BatchNormalization(),
        Conv2D(32, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        # Block 2
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        # Block 3
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        # Classifier
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    return model

cnn_model = build_cnn()
cnn_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
cnn_model.summary()

callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1)
]

print("\nTraining CNN (max 20 epochs)...")
history_cnn = cnn_model.fit(
    datagen.flow(x_train_n, y_train_oh, batch_size=64),
    steps_per_epoch = len(x_train_n) // 64,
    validation_data = (x_test_n, y_test_oh),
    epochs          = 20,
    callbacks       = callbacks_cnn,
    verbose         = 1
)

cnn_loss, cnn_acc = cnn_model.evaluate(
    x_test_n, y_test_oh, verbose=0)
cnn_pred_prob = cnn_model.predict(x_test_n, verbose=0)
cnn_pred      = cnn_pred_prob.argmax(axis=1)
print(f"\nCNN Accuracy : {cnn_acc:.4f}")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("BLOCK 3 — CNN Training Curves",
             fontsize=13, fontweight='bold')

axes[0].plot(history_cnn.history['accuracy'],
             label='Train', color='steelblue')
axes[0].plot(history_cnn.history['val_accuracy'],
             label='Val',   color='coral')
axes[0].set_title("Accuracy per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_cnn.history['loss'],
             label='Train', color='steelblue')
axes[1].plot(history_cnn.history['val_loss'],
             label='Val',   color='coral')
axes[1].set_title("Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("cnn_training.png", dpi=120, bbox_inches='tight')
plt.show()
print(" Block 3 done")


# BLOCK 4 — TRANSFER LEARNING (MobileNetV2)

print("\n" + "="*60)
print("BLOCK 4 — Transfer Learning (MobileNetV2)")
print("="*60)

N_TL = 10000
print(f"Resizing {N_TL} images to 96x96 for MobileNetV2...")
x_train_96 = tf.image.resize(
    x_train_n[:N_TL], [96, 96]).numpy()
x_test_96  = tf.image.resize(
    x_test_n[:2000],  [96, 96]).numpy()

y_train_tl = y_train_oh[:N_TL]
y_test_tl  = y_test_oh[:2000]
print(f"x_train_96 : {x_train_96.shape}")

# Build MobileNetV2 model
base_model = MobileNetV2(
    input_shape = (96, 96, 3),
    include_top = False,
    weights     = 'imagenet'
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=(96, 96, 3))
x       = base_model(inputs, training=False)
x       = GlobalAveragePooling2D()(x)
x       = Dense(128, activation='relu')(x)
x       = Dropout(0.3)(x)
outputs = Dense(10,  activation='softmax')(x)

tl_model = Model(inputs, outputs)
tl_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

trainable_params = sum(
    [np.prod(v.shape) for v in tl_model.trainable_variables])
print(f"Trainable parameters : {trainable_params:,}")

callbacks_tl = [
    EarlyStopping(monitor='val_accuracy', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, verbose=1)
]

print("\nTraining Transfer Learning model (max 15 epochs)...")
history_tl = tl_model.fit(
    x_train_96, y_train_tl,
    validation_data = (x_test_96, y_test_tl),
    epochs          = 15,
    batch_size      = 64,
    callbacks       = callbacks_tl,
    verbose         = 1
)

tl_loss, tl_acc = tl_model.evaluate(
    x_test_96, y_test_tl, verbose=0)
tl_pred = tl_model.predict(
    x_test_96, verbose=0).argmax(axis=1)
print(f"\nMobileNetV2 TL Accuracy : {tl_acc:.4f}")
print(" Block 4 done")


# BLOCK 5 — COMPARISON & FINAL VISUALIZATION

print("\n" + "="*60)
print("BLOCK 5 — Comparison & Final Visualization")
print("="*60)

methods_img = ['SVM\n(HOG+PCA)', 'CNN\nCustom', 'MobileNetV2\n(TL)']
accs_img    = [svm_acc, cnn_acc, tl_acc]
colors_img  = ['steelblue', 'coral', 'purple']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("BLOCK 5 — Image Analytics Final Comparison",
             fontsize=14, fontweight='bold')

# 5A. Accuracy comparison
bars = axes[0,0].bar(methods_img, accs_img,
                      color=colors_img,
                      alpha=0.85, edgecolor='white')
axes[0,0].set_ylim(0, 1.1)
axes[0,0].set_title("Accuracy Comparison — All Methods")
axes[0,0].set_ylabel("Accuracy")
for bar, acc in zip(bars, accs_img):
    axes[0,0].text(
        bar.get_x() + bar.get_width()/2,
        acc + 0.01, f"{acc:.3f}",
        ha='center', fontsize=11, fontweight='bold')

# 5B. CNN Confusion Matrix
cm_cnn = confusion_matrix(y_test_flat, cnn_pred)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            ax=axes[0,1],
            xticklabels=CIFAR_LABELS,
            yticklabels=CIFAR_LABELS)
axes[0,1].set_title(f"CNN Confusion Matrix\nAcc={cnn_acc:.3f}")
axes[0,1].tick_params(axis='x', rotation=45, labelsize=7)
axes[0,1].tick_params(axis='y', rotation=0,  labelsize=7)

# 5C. MobileNetV2 Confusion Matrix
cm_tl = confusion_matrix(y_test_flat[:2000], tl_pred)
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Purples',
            ax=axes[0,2],
            xticklabels=CIFAR_LABELS,
            yticklabels=CIFAR_LABELS)
axes[0,2].set_title(f"MobileNetV2 Confusion Matrix\nAcc={tl_acc:.3f}")
axes[0,2].tick_params(axis='x', rotation=45, labelsize=7)
axes[0,2].tick_params(axis='y', rotation=0,  labelsize=7)

# 5D. Per-class accuracy CNN
acc_per_class = []
for cls in range(10):
    mask    = y_test_flat == cls
    cls_acc = accuracy_score(
        y_test_flat[mask], cnn_pred[mask])
    acc_per_class.append(cls_acc)

axes[1,0].bar(CIFAR_LABELS, acc_per_class,
              color='steelblue', alpha=0.8,
              edgecolor='white')
axes[1,0].set_title("CNN — Per-class Accuracy")
axes[1,0].set_ylabel("Accuracy")
axes[1,0].tick_params(axis='x', rotation=30)
axes[1,0].set_ylim(0, 1.1)
for i, acc in enumerate(acc_per_class):
    axes[1,0].text(i, acc+0.01, f"{acc:.2f}",
                   ha='center', fontsize=8)

# 5E. Prediction examples
fig2, ax2 = plt.subplots(2, 8, figsize=(18, 5))
fig2.suptitle(
    "CNN Predictions — Green=Correct / Red=Incorrect",
    fontsize=12)
correct_idx   = np.where(cnn_pred == y_test_flat)[0][:8]
incorrect_idx = np.where(cnn_pred != y_test_flat)[0][:8]

for i, idx in enumerate(correct_idx):
    ax2[0,i].imshow(x_test_n[idx])
    ax2[0,i].set_title(
        f"✓{CIFAR_LABELS[cnn_pred[idx]]}",
        fontsize=7, color='green')
    ax2[0,i].axis('off')

for i, idx in enumerate(incorrect_idx):
    ax2[1,i].imshow(x_test_n[idx])
    ax2[1,i].set_title(
        f"✗{CIFAR_LABELS[cnn_pred[idx]]}\n"
        f"({CIFAR_LABELS[y_test_flat[idx]]})",
        fontsize=6, color='red')
    ax2[1,i].axis('off')

plt.tight_layout()
plt.savefig("cnn_predictions.png", dpi=120, bbox_inches='tight')
plt.show()

# 5F. Summary table
recap_img = pd.DataFrame({
    'Method'    : ['SVM (HOG+PCA)',
                   'Custom CNN',
                   'MobileNetV2 TL'],
    'Accuracy'  : [f"{svm_acc:.4f}",
                   f"{cnn_acc:.4f}",
                   f"{tl_acc:.4f}"],
    'Train Size': [N_HOG, 50000, N_TL],
    'Type'      : ['Classical ML',
                   'Deep Learning',
                   'Transfer Learning'],
})
axes[1,1].axis('off')
tbl = axes[1,1].table(
    cellText  = recap_img.values,
    colLabels = recap_img.columns,
    cellLoc   = 'center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 2.2)
axes[1,1].set_title("Methods Summary", pad=20)

# TL training curve
axes[1,2].plot(history_tl.history['accuracy'],
               label='Train', color='steelblue', lw=2)
axes[1,2].plot(history_tl.history['val_accuracy'],
               label='Val',   color='coral', lw=2)
axes[1,2].set_title("MobileNetV2 — Accuracy per Epoch")
axes[1,2].set_xlabel("Epoch")
axes[1,2].legend()

plt.tight_layout()
plt.savefig("image_analytics_comparison.png",
            dpi=120, bbox_inches='tight')
plt.show()

# ── Final Summary Section 7
print("\n" + "="*60)
print(" SECTION 7 COMPLETE")
print("="*60)
print(f"\n  SVM (HOG+PCA)  : Accuracy = {svm_acc:.4f}")
print(f"  Custom CNN     : Accuracy = {cnn_acc:.4f}")
print(f"  MobileNetV2 TL : Accuracy = {tl_acc:.4f}")
print("\n  Saved files :")
print("   cifar_preprocessing.png")
print("   hog_features.png")
print("   pca_variance.png")
print("   cnn_training.png")
print("   cnn_predictions.png")
print("   image_analytics_comparison.png")
print("\n Ready for Section 8 — Multimodal + Final Dashboard")